# Эксперименты: препроцессинг и словари

Проверяем, что дают функции `preprocess_raw_csv` и `build_vocab_mappings` на срезе данных.

In [1]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

## 1. Загрузка среза данных (первые N строк)

Читаем первые 1000 строк, чтобы было достаточно уникальных игроков и команд для демо. Сохраняем в отдельный CSV, чтобы не перезаписывать основной processed.

In [2]:
N_ROWS = 1000
raw_path = ROOT / "dataset" / "data_with_dates.csv"
sample_path = ROOT / "notebooks" / "sample_raw.csv"

df_raw = pd.read_csv(raw_path, nrows=N_ROWS)
df_raw.to_csv(sample_path, index=False)
print(f"Загружено строк: {len(df_raw)}")
df_raw.head(10)

Загружено строк: 1000


,player_name,team_name,competition_name,season_name,match_id,match_date,is_aligned,position_id,position_name,pass_total,...,foul_won_total,foul_won_penalty,foul_committed_total,foul_committed_penalty,foul_committed_yellow_card,foul_committed_red_card,goalkeeper_goal_conceded,goalkeeper_save,goalkeeper_shot_faced,counterpress_total
0,Lukáš Hrádecký,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,1,Goalkeeper,24,...,0,0,0,0,0,0,0,0,0,0
1,Odilon Kossonou,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,3,Right Center Back,77,...,0,0,1,0,0,0,0,0,0,0
2,Jonathan Tah,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,4,Center Back,74,...,0,0,1,0,0,0,0,0,0,0
3,Edmond Fayçal Tapsoba,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,5,Left Center Back,87,...,0,0,4,0,0,0,0,0,0,0
4,Nathan Tella,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,7,Right Wing Back,24,...,1,0,0,0,0,0,0,0,0,0
5,Piero Martín Hincapié Reyna,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,8,Left Wing Back,42,...,0,0,0,0,0,0,0,0,0,0
6,Granit Xhaka,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,9,Right Defensive Midfield,85,...,1,0,1,0,0,0,0,0,0,0
7,Robert Andrich,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,11,Left Defensive Midfield,88,...,2,0,1,0,0,0,0,0,0,0
8,Jonas Hofmann,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,18,Right Attacking Midfield,46,...,1,1,0,0,0,0,0,0,0,0
9,Amine Adli,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,20,Left Attacking Midfield,21,...,1,0,2,0,1,0,0,0,0,0


## 2. preprocess_raw_csv

Запускаем препроцессинг: валидация колонок, заполнение NaN в статах нулями, приведение `position_id` и `match_date`, сохранение в `notebooks/processed_sample/processed.csv`.

In [3]:
from data.preprocessing import preprocess_raw_csv

output_dir = str(ROOT / "notebooks" / "processed_sample")
df_processed = preprocess_raw_csv(str(sample_path), output_dir)

print("Shape:", df_processed.shape)
print("\nКолонки:", list(df_processed.columns))
print("\nТипы:")
df_processed.dtypes

Shape: (1000, 48)

Колонки: ['player_name', 'team_name', 'competition_name', 'season_name', 'match_id', 'match_date', 'is_aligned', 'position_id', 'position_name', 'pass_total', 'pass_cross', 'pass_cut_back', 'pass_shot_assist', 'pass_goal_assist', 'pass_no_touch', 'pass_interception', 'pass_incomplete', 'pass_offside', 'pass_through_ball', 'shot_total', 'shot_statsbomb_xg', 'shot_corner', 'shot_free_kick', 'shot_open_play', 'shot_penalty', 'shot_saved', 'shot_off_target', 'shot_blocked', 'shot_goal', 'interception_total', 'interception_won', 'interception_lost', 'block_total', 'clearance_total', 'ball_recovery_total', 'dribble_total', 'dribble_complete', 'dribble_incomplete', 'foul_won_total', 'foul_won_penalty', 'foul_committed_total', 'foul_committed_penalty', 'foul_committed_yellow_card', 'foul_committed_red_card', 'goalkeeper_goal_conceded', 'goalkeeper_save', 'goalkeeper_shot_faced', 'counterpress_total']

Типы:


player_name                              str
team_name                                str
competition_name                         str
season_name                              str
match_id                               int64
match_date                    datetime64[us]
is_aligned                             int64
position_id                            int64
position_name                            str
pass_total                             int64
pass_cross                             int64
pass_cut_back                          int64
pass_shot_assist                       int64
pass_goal_assist                       int64
pass_no_touch                          int64
pass_interception                      int64
pass_incomplete                        int64
pass_offside                           int64
pass_through_ball                      int64
shot_total                             int64
shot_statsbomb_xg                    float64
shot_corner                            int64
shot_free_

In [4]:
# Первые 100 строк обработанных данных
display(df_processed.head(100))

,player_name,team_name,competition_name,season_name,match_id,match_date,is_aligned,position_id,position_name,pass_total,...,foul_won_total,foul_won_penalty,foul_committed_total,foul_committed_penalty,foul_committed_yellow_card,foul_committed_red_card,goalkeeper_goal_conceded,goalkeeper_save,goalkeeper_shot_faced,counterpress_total
0,Lukáš Hrádecký,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,1,Goalkeeper,24,...,0,0,0,0,0,0,0,0,0,0
1,Odilon Kossonou,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,3,Right Center Back,77,...,0,0,1,0,0,0,0,0,0,0
2,Jonathan Tah,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,4,Center Back,74,...,0,0,1,0,0,0,0,0,0,0
3,Edmond Fayçal Tapsoba,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,5,Left Center Back,87,...,0,0,4,0,0,0,0,0,0,0
4,Nathan Tella,Bayer Leverkusen,1. Bundesliga,2023/2024,3895302,2024-04-14,1,7,Right Wing Back,24,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Exequiel Alejandro Palacios,Bayer Leverkusen,1. Bundesliga,2023/2024,3895348,2024-05-18,1,11,Left Defensive Midfield,72,...,3,0,1,0,0,0,0,0,0,0
96,Jonas Hofmann,Bayer Leverkusen,1. Bundesliga,2023/2024,3895348,2024-05-18,1,18,Right Attacking Midfield,66,...,0,0,0,0,0,0,0,0,0,0
97,Amine Adli,Bayer Leverkusen,1. Bundesliga,2023/2024,3895348,2024-05-18,1,20,Left Attacking Midfield,36,...,3,0,0,0,0,0,0,0,0,0
98,Victor Okoh Boniface,Bayer Leverkusen,1. Bundesliga,2023/2024,3895348,2024-05-18,1,23,Center Forward,19,...,0,0,1,0,0,0,0,0,0,0


In [5]:
# Пример стат по матчу (одна команда)
mask = (df_processed["match_id"] == df_processed["match_id"].iloc[0]) & (df_processed["team_name"] == df_processed["team_name"].iloc[0])
df_processed.loc[mask, ["player_name", "position_name", "pass_total", "shot_total", "goalkeeper_save"]].head(15)

,player_name,position_name,pass_total,shot_total,goalkeeper_save
0,Lukáš Hrádecký,Goalkeeper,24,0,0
1,Odilon Kossonou,Right Center Back,77,0,0
2,Jonathan Tah,Center Back,74,3,0
3,Edmond Fayçal Tapsoba,Left Center Back,87,1,0
4,Nathan Tella,Right Wing Back,24,0,0
5,Piero Martín Hincapié Reyna,Left Wing Back,42,1,0
6,Granit Xhaka,Right Defensive Midfield,85,3,0
7,Robert Andrich,Left Defensive Midfield,88,0,0
8,Jonas Hofmann,Right Attacking Midfield,46,1,0
9,Amine Adli,Left Attacking Midfield,21,1,0


## 3. build_vocab_mappings

Строим словари имя↔id для игроков и команд, спец. токены (mask, pad), сохраняем pickle в `notebooks/processed_sample/metadata/`.

In [6]:
from data.preprocessing import build_vocab_mappings

vocab = build_vocab_mappings(df_processed, output_dir)

print("Размеры словарей:")
print("  players_vocab_size:", vocab["players_vocab_size"])
print("  teams_vocab_size:  ", vocab["teams_vocab_size"])
print("\nСпец. токены:")
print("  player_mask_token_id:", vocab["player_mask_token_id"])
print("  player_pad_token_id: ", vocab["player_pad_token_id"])
print("  team_pad_token_id:   ", vocab["team_pad_token_id"])

Размеры словарей:
  players_vocab_size: 432
  teams_vocab_size:   23

Спец. токены:
  player_mask_token_id: 430
  player_pad_token_id:  431
  team_pad_token_id:    22


In [7]:
# Пример: имя игрока → id (для эмбеддинга PE)
player_items = list(vocab["player_name2id"].items())[:15]
pd.DataFrame(player_items, columns=["player_name", "player_id"])

,player_name,player_id
0,Adam Hložek,0
1,Adelino André Vieira Freitas,1
2,Adrian Beck,2
3,Alassane Pléa,3
4,Albin Ekdal,4
5,Alejandro Grimaldo García,5
6,Aleksandar Ignjovski,6
7,Aleksandar Pavlović,7
8,Alex Král,8
9,Alexander Meier,9


In [8]:
# Обратный маппинг: id → имя игрока
id2name = vocab["id2player_name"]
pd.DataFrame([(i, id2name[i]) for i in range(min(10, len(id2name)))], columns=["player_id", "player_name"])

,player_id,player_name
0,0,Adam Hložek
1,1,Adelino André Vieira Freitas
2,2,Adrian Beck
3,3,Alassane Pléa
4,4,Albin Ekdal
5,5,Alejandro Grimaldo García
6,6,Aleksandar Ignjovski
7,7,Aleksandar Pavlović
8,8,Alex Král
9,9,Alexander Meier


In [9]:
# Команды: имя → id (для эмбеддинга TE)
team_items = list(vocab["team_name2id"].items())
pd.DataFrame(team_items, columns=["team_name", "team_id"])

,team_name,team_id
0,Augsburg,0
1,Bayer Leverkusen,1
2,Bayern Munich,2
3,Bochum,3
4,Borussia Dortmund,4
5,Borussia Mönchengladbach,5
6,Darmstadt 98,6
7,Eintracht Frankfurt,7
8,FC Heidenheim,8
9,FC Köln,9


## 4. Проверка: загрузка словарей из pickle

Убеждаемся, что сохранённые в `metadata/` словари читаются обратно.

In [10]:
from data.preprocessing import _load_pickle

meta_dir = Path(output_dir) / "metadata"
loaded_player2id = _load_pickle(str(meta_dir / "player_name2id.pickle"))
loaded_team2id = _load_pickle(str(meta_dir / "team_name2id.pickle"))

print("Загружено player_name2id:", len(loaded_player2id), "записей")
print("Загружено team_name2id:", len(loaded_team2id), "записей")
print("Совпадает с vocab:", loaded_player2id == vocab["player_name2id"], loaded_team2id == vocab["team_name2id"])

Загружено player_name2id: 430 записей
Загружено team_name2id: 22 записей
Совпадает с vocab: True True


In [11]:
# Пример: преобразовать несколько имён в id для батча
names = df_processed["player_name"].head(5).tolist()
ids = [vocab["player_name2id"][n] for n in names]
pd.DataFrame({"player_name": names, "player_id": ids})

,player_name,player_id
0,Lukáš Hrádecký,238
1,Odilon Kossonou,313
2,Jonathan Tah,177
3,Edmond Fayçal Tapsoba,89
4,Nathan Tella,295


## 5. MatchDatasetMPP — датасет для Masked Player Prediction

Один элемент = один матч: последовательность игроков (обе команды), закодированная в тензоры. Маскирование делается в коллаторе, здесь только паддинг до `max_seq_length`.

In [12]:
from data.dataset import MatchDatasetMPP

max_seq_length = 36
ds_mpp = MatchDatasetMPP(
    df_processed,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

print("Число матчей (элементов датасета):", len(ds_mpp))

Число матчей (элементов датасета): 46


In [13]:
# Один элемент — один матч
sample = ds_mpp[0]
print("Ключи одного элемента:", list(sample.keys()))
for k, v in sample.items():
    print(f"  {k}: shape = {v.shape}, dtype = {v.dtype}")

Ключи одного элемента: ['input_ids', 'position_id', 'team_id', 'form_stats', 'attention_mask']
  input_ids: shape = torch.Size([36]), dtype = torch.int64
  position_id: shape = torch.Size([36]), dtype = torch.int64
  team_id: shape = torch.Size([36]), dtype = torch.int64
  form_stats: shape = torch.Size([36, 39]), dtype = torch.float32
  attention_mask: shape = torch.Size([36]), dtype = torch.int64


In [14]:
# Расшифровка: сколько реальных игроков в первом матче
n_real = sample["attention_mask"].sum().item()
print(f"В первом матче игроков (без паддинга): {n_real}")
id2player = vocab["id2player_name"]
player_ids = sample["input_ids"][:n_real].tolist()
names = [id2player.get(i, f"<id={i}>") for i in player_ids]
pd.DataFrame({"position": range(n_real), "player_id": player_ids, "player_name": names})

В первом матче игроков (без паддинга): 22


,position,player_id,player_name
0,0,238,Lukáš Hrádecký
1,1,313,Odilon Kossonou
2,2,177,Jonathan Tah
3,3,89,Edmond Fayçal Tapsoba
4,4,295,Nathan Tella
5,5,335,Piero Martín Hincapié Reyna
6,6,135,Granit Xhaka
7,7,348,Robert Andrich
8,8,173,Jonas Hofmann
9,9,15,Amine Adli


In [15]:
# form_stats: 39 стат на каждого игрока (TPE в модели)
print("form_stats: форма (max_seq_length, 39) =", sample["form_stats"].shape)
print("Пример: первые 5 стат у первых 3 игроков:")
pd.DataFrame(
    sample["form_stats"][:3, :5].numpy(),
    columns=["pass_total", "pass_cross", "pass_cut_back", "pass_shot_assist", "pass_goal_assist"],
    index=[names[i] for i in range(3)],
)

form_stats: форма (max_seq_length, 39) = torch.Size([36, 39])
Пример: первые 5 стат у первых 3 игроков:


,pass_total,pass_cross,pass_cut_back,pass_shot_assist,pass_goal_assist
Lukáš Hrádecký,24.0,0.0,0.0,0.0,0.0
Odilon Kossonou,77.0,0.0,0.0,0.0,0.0
Jonathan Tah,74.0,0.0,0.0,0.0,0.0


In [16]:
# Несколько матчей подряд (без DataLoader — просто цикл)
for i in range(min(3, len(ds_mpp))):
    item = ds_mpp[i]
    if item is None:
        print(f"  Матч {i}: пропущен (не 2 команды)")
    else:
        n = item["attention_mask"].sum().item()
        print(f"  Матч {i}: игроков = {n}, input_ids shape = {item['input_ids'].shape}")

  Матч 0: игроков = 22, input_ids shape = torch.Size([36])
  Матч 1: игроков = 22, input_ids shape = torch.Size([36])
  Матч 2: игроков = 22, input_ids shape = torch.Size([36])


## 6. DataCollatorMPP — маскирование и сборка батча

Коллатор берёт список матчей (от MatchDatasetMPP), в каждом матче случайно маскирует ~25% игроков (подставляет `player_mask_token_id`, обнуляет их статы), строит тензор `labels` (исходный id в замаскированных позициях, −100 в остальных) и склеивает всё в один батч.

In [17]:
from data.collator import DataCollatorMPP

collator = DataCollatorMPP(
    player_mask_token_id=vocab["player_mask_token_id"],
    mask_percentage=0.25,
)

# Собираем «батч» из нескольких матчей вручную (как делает DataLoader)
batch_matches = [ds_mpp[i] for i in range(4) if ds_mpp[i] is not None]
batch = collator(batch_matches)

print("После коллатора — один батч:")
for k, v in batch.items():
    print(f"  {k}: shape = {v.shape}")

После коллатора — один батч:
  input_ids: shape = torch.Size([4, 36])
  labels: shape = torch.Size([4, 36])
  position_id: shape = torch.Size([4, 36])
  team_id: shape = torch.Size([4, 36])
  form_stats: shape = torch.Size([4, 36, 39])
  attention_mask: shape = torch.Size([4, 36])


In [18]:
# Где маска: input_ids == mask_token, в labels — истинный player id
mask_token = vocab["player_mask_token_id"]
is_masked = batch["labels"] != -100
print("Всего замаскированных позиций в батче:", is_masked.sum().item())
print("В этих позициях input_ids == mask_token:", (batch["input_ids"][is_masked] == mask_token).all().item())
print("В этих позициях labels = исходный player id (не -100):", (batch["labels"][is_masked] >= 0).all().item())

Всего замаскированных позиций в батче: 24
В этих позициях input_ids == mask_token: True
В этих позициях labels = исходный player id (не -100): True


In [19]:
# Первый матч в батче: позиция, исходный id (из labels), замаскированный input_id
match_idx = 0
labels_m0 = batch["labels"][match_idx]
input_m0 = batch["input_ids"][match_idx]
n_real = batch["attention_mask"][match_idx].sum().item()
id2player = vocab["id2player_name"]
rows = []
for pos in range(n_real):
    orig_id = labels_m0[pos].item()
    masked_id = input_m0[pos].item()
    rows.append({
        "позиция": pos,
        "input_id (после маски)": masked_id,
        "label (кого угадывать)": orig_id if orig_id != -100 else "—",
        "имя игрока": id2player.get(orig_id, id2player.get(masked_id, "?")) if orig_id != -100 else id2player.get(masked_id, "?"),
    })
pd.DataFrame(rows)

,позиция,input_id (после маски),label (кого угадывать),имя игрока
0,0,430,238,Lukáš Hrádecký
1,1,313,—,Odilon Kossonou
2,2,177,—,Jonathan Tah
3,3,89,—,Edmond Fayçal Tapsoba
4,4,295,—,Nathan Tella
5,5,335,—,Piero Martín Hincapié Reyna
6,6,135,—,Granit Xhaka
7,7,348,—,Robert Andrich
8,8,173,—,Jonas Hofmann
9,9,15,—,Amine Adli


In [20]:
# Замаскированные позиции в первом матче: модель должна предсказать именно этих игроков
masked_pos = (batch["labels"][0] != -100).nonzero(as_tuple=True)[0]
print("Замаскированные слоты в матче 0:", masked_pos.tolist())
print("Их истинные id (target):", batch["labels"][0][masked_pos].tolist())
print("Имена (цель MPP):", [vocab["id2player_name"][batch["labels"][0][p].item()] for p in masked_pos])

Замаскированные слоты в матче 0: [0, 10, 12, 13, 17, 18]
Их истинные id (target): [238, 412, 192, 49, 373, 221]
Имена (цель MPP): ['Lukáš Hrádecký', 'Victor Okoh Boniface', 'Julián Malatini', 'Christian Groß', 'Senne Lynen', 'Leonardo Bittencourt']


## 7. PlayerEncoder — продолжение пайплайна

Батч из коллатора подаём в энкодер: он строит PE + SPE + TPE (+ TE), прогоняет через трансформер и возвращает контекстные представления игроков `(batch, seq_len, embed_size)` и матрицы внимания.

In [23]:
from models.encoder import PlayerEncoder

embed_size = 64
encoder = PlayerEncoder(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],  # pad_idx = этот же индекс
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=True,
    position_enc_type="learned",
)

# Батч с коллатора — те же ключи, что ожидает encoder.forward
player_repr, attention_matrices = encoder(
    player_id=batch["input_ids"],
    position_id=batch["position_id"],
    team_id=batch["team_id"],
    form_stats=batch["form_stats"],
    attention_mask=batch["attention_mask"],
)

print("Выход энкодера:")
print("  player_representations:", player_repr.shape)
print("  attention_matrices: список из", len(attention_matrices), "тензоров")
if attention_matrices:
    print("  одна матрица внимания:", attention_matrices[0].shape)

Выход энкодера:
  player_representations: torch.Size([4, 36, 64])
  attention_matrices: список из 1 тензоров
  одна матрица внимания: torch.Size([4, 2, 36, 36])


In [25]:
# Пример: представление первого матча, первые 5 игроков (слоты 0–4)
# Это вход для MPP-головы: по этим векторам предсказываем замаскированных
first_match_repr = player_repr[0, :5, :].detach()
n_show = min(8, embed_size)
pd.DataFrame(
    first_match_repr.numpy().round(4)[:, :n_show],
    index=[f"игрок_{i}" for i in range(5)],
    columns=[f"dim_{j}" for j in range(n_show)],
)

,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7
игрок_0,-0.2543,1.7157,-0.8399,0.1039,1.0617,0.8573,-0.9056,0.9887
игрок_1,-0.4431,-0.6666,-1.3217,-0.0000,-0.2225,2.0705,-1.1181,-0.3588
игрок_2,-0.6055,-0.6050,-1.2026,-1.3305,-0.4464,2.4900,-1.1777,-0.4300
игрок_3,-0.6735,-0.4334,-1.1122,-1.1785,-0.4125,0.0490,-1.1036,-0.4687
игрок_4,-0.1553,0.2402,-1.1697,-0.0000,0.2583,0.1916,-1.0542,0.0782


In [26]:
# Все четыре типа эмбеддингов в энкодере (для анализа после обучения)
# PE — кто игрок (player_id → вектор)
W_player = encoder.get_player_embeddings_weight()
print("PE (player embeddings):   ", W_player.shape, "— игроки + mask + pad")

# SPE — позиция на поле (position_id → вектор)
W_pos = encoder.get_position_embeddings_weight()
print("SPE (position embeddings):", W_pos.shape, "— 25 позиций + pad")

# TPE — статы матча (39 чисел → вектор через Linear)
print("TPE (form/stats → Linear):", encoder.form_embeddings.weight.shape, "(39 → embed_size)")

# TE — команда (team_id → вектор), если включено
if encoder.use_teams_embeddings:
    W_team = encoder.teams_embeddings.weight
    print("TE (team embeddings):     ", W_team.shape, "— команды + pad")
else:
    print("TE (team embeddings):      не используются (use_teams_embeddings=False)")

Таблица эмбеддингов игроков (PE): torch.Size([432, 64])
Таблица эмбеддингов позиций (SPE): torch.Size([26, 64])
